In [1]:
# === SESSION BOOTSTRAP ===
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
PARENT="/content/drive/MyDrive/UAV_TRUST_Research"; REPO=f"{PARENT}/uav-trust-research"
for fn in (".gitconfig",".git-credentials"):
    p=os.path.join(PARENT,fn)
    if os.path.exists(p): subprocess.run(f'cp "{p}" /root/{fn}',shell=True)
subprocess.run("git config --global credential.helper store",shell=True)
if os.path.isdir(REPO):
    os.chdir(REPO); sys.path.insert(0,REPO) if REPO not in sys.path else None; print("cwd:",os.getcwd())
else: print("run 00_setup first")

Mounted at /content/drive
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install xgboost shap scikit-learn matplotlib pandas numpy scipy requests --quiet

In [3]:
# UAV-CAS (classical, 47 flow features) as a THIRD validation dataset.
# Attack-family-held-out protocol, six single-family classes, collaborative attacks set aside.
CAS_STAT="/content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research/data/uav_cas/UAV-CAS_stat.csv"
SINGLE={"benign","dos","ddos","blackhole","wormhole","replay"}   # matched case-insensitively; "+"-labels excluded
CAS={"label_col":"Label","normal_key":"benign",
     "drop_meta":["config_idx","num_drones","num_bs","payload","pathloss","modulation","mission","tx_power","noise","src_ip","dst_ip"]}
CFG={"seeds":list(range(10)),"alpha":0.10,"nbins":15,"n_shap":1500,"drop_fraction":0.20,
     "normal_fracs":{"train":0.60,"cal":0.20,"test_seen":0.10,"test_shift":0.10},
     "family_fracs":{"train":0.60,"cal":0.20,"test_seen":0.20},
     "xgb":{"n_estimators":300,"max_depth":6,"learning_rate":0.1,"subsample":0.9,"colsample_bytree":0.9,"tree_method":"hist"},
     "fig_dir":"figures","report_dir":"reports"}
for d in [CFG["fig_dir"],CFG["report_dir"]]: os.makedirs(d,exist_ok=True)
print("configured")

configured


In [4]:
import numpy as np, pandas as pd, importlib, gc, src.data
importlib.reload(src.data)
import matplotlib.pyplot as plt, xgboost as xgb, shap
from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from src.trust import top_label_ece, brier_binary, conformal_qhat, aurc, logit, fit_calibrators, apply_calibrators
from src.tsfs import mean_shap, reversal_vector, internal_fragility

# ---- load + INSPECT labels (prints class counts so you can sanity-check spelling) ----
raw=pd.read_csv(CAS_STAT, low_memory=False)
print("raw rows:", len(raw), "cols:", raw.shape[1])
print("\nALL labels and counts (single-family vs '+'-joined collaborative):")
print(raw[CAS["label_col"]].value_counts().to_string())
lab=raw[CAS["label_col"]].astype(str)
keep = lab.str.lower().isin(SINGLE) & (~lab.str.contains("+", regex=False))
df=raw[keep].reset_index(drop=True)
normal_value=[v for v in df[CAS["label_col"]].unique() if str(v).lower()==CAS["normal_key"]][0]
fams=[v for v in df[CAS["label_col"]].unique() if v!=normal_value]
print("\nAFTER six-class filter -> rows:", len(df), "| normal label:", repr(normal_value))
print("attack families kept:", fams)

raw rows: 99492 cols: 59

ALL labels and counts (single-family vs '+'-joined collaborative):
Label
Benign                58589
Blackhole             12116
Wormhole              10862
DDoS                   7316
Blackhole+DDoS         2493
Replay                 2025
DDoS+Wormhole          1871
Blackhole+Wormhole     1332
DoS                    1223
DDoS+Replay             536
DoS+Wormhole            396
Blackhole+DoS           293
Blackhole+Replay        185
Replay+Wormhole         155
DoS+Replay              100

AFTER six-class filter -> rows: 92131 | normal label: 'Benign'
attack families kept: ['DoS', 'DDoS', 'Blackhole', 'Wormhole', 'Replay']


In [5]:
# Clean feature matrix (drop metadata + address columns by name), then attack-family-held-out splits.
def cas_prepare(df, held_out, seed):
    lc=CAS["label_col"]; nv=normal_value
    feat=df.drop(columns=[col for col in CAS["drop_meta"] if col in df.columns]+[lc]).copy()
    feat=feat.apply(pd.to_numeric, errors="coerce")
    const=[col for col in feat.columns if feat[col].nunique(dropna=False)<=1]; feat=feat.drop(columns=const)
    feat=feat.replace([np.inf,-np.inf],np.nan).fillna(0.0)
    X=feat.values.astype(np.float32); y=(df[lc].values!=nv).astype(int); fam=df[lc].values
    seen=[f for f in fams if f!=held_out]
    def sp(idx,fr,sd):
        idx=np.array(idx); r=np.random.default_rng(sd); r.shuffle(idx); out={}; s=0; ks=list(fr)
        for i,k in enumerate(ks):
            e=len(idx) if i==len(ks)-1 else s+int(round(fr[k]*len(idx))); out[k]=idx[s:e]; s=e
        return out
    tr=[];ca=[];se=[];sh=[]
    ns=sp(np.where(fam==nv)[0],CFG["normal_fracs"],seed); tr+=list(ns["train"]);ca+=list(ns["cal"]);se+=list(ns["test_seen"]);sh+=list(ns["test_shift"])
    for j,f in enumerate(seen):
        fs=sp(np.where(fam==f)[0],CFG["family_fracs"],seed+j+1); tr+=list(fs["train"]);ca+=list(fs["cal"]);se+=list(fs["test_seen"])
    sh+=list(np.where(fam==held_out)[0])
    tr,ca,se,sh=(np.array(sorted(a)) for a in (tr,ca,se,sh))
    sc=StandardScaler().fit(X[tr]); tf=lambda ix: sc.transform(X[ix]).astype(np.float32)
    return {"held_out":held_out,"seen_families":seen,"feature_names":list(feat.columns),
            "X_train":tf(tr),"y_train":y[tr],"fam_train":fam[tr],"X_cal":tf(ca),"y_cal":y[ca],
            "X_seen":tf(se),"y_seen":y[se],"fam_seen":fam[se],"X_shift":tf(sh),"y_shift":y[sh],"fam_shift":fam[sh]}
S0=cas_prepare(df,fams[0],0); print("feature count after cleaning:",S0["X_train"].shape[1])
print("first features:",sorted(S0["feature_names"])[:6],"..."); del S0; gc.collect()

feature count after cleaning: 44
first features: ['ACK Flag Count', 'Average Packet Size', 'Bwd Header Length', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Total'] ...


0

In [6]:
# Trust panel per (family, seed): split calibration, conformal on raw scores, full metrics + set decomposition
fit=lambda X,y,sd: xgb.XGBClassifier(objective="binary:logistic",eval_metric="logloss",random_state=sd,**CFG["xgb"]).fit(X,y)
def cov_sets(p,y,qh):
    p=np.asarray(p);y=np.asarray(y);ia=(1-p)<=qh;ino=p<=qh
    inset=np.where(y==1,ia,ino);size=ia.astype(int)+ino.astype(int)
    return dict(marg=float(inset.mean()),bal=float(np.mean([inset[y==k].mean() for k in np.unique(y)])),
                size=float(size.mean()),empty=float(((~ia)&(~ino)).mean()),
                normal_only=float((ino&(~ia)).mean()),attack_only=float((ia&(~ino)).mean()),both=float((ia&ino).mean()))
rows=[]
for F in fams:
    for seed in CFG["seeds"]:
        S=cas_prepare(df,F,seed); clf=fit(S["X_train"],S["y_train"],seed)
        Xc,yc=S["X_cal"],S["y_cal"]; perm=np.random.default_rng(seed).permutation(len(yc)); h=len(yc)//2; ip,ic=perm[:h],perm[h:]
        pr=lambda X: clf.predict_proba(X)[:,1]
        p_pc=pr(Xc[ip]); p_cc=pr(Xc[ic]); p_se=pr(S["X_seen"]); p_sh=pr(S["X_shift"]); ys,yh=S["y_seen"],S["y_shift"]
        fc=fit_calibrators(logit(p_pc),p_pc,yc[ip]); ch=apply_calibrators(fc,logit(p_sh),p_sh); cs=apply_calibrators(fc,logit(p_se),p_se)
        qh=conformal_qhat(p_cc,yc[ic],alpha=CFG["alpha"]); sh=cov_sets(p_sh,yh,qh); se=cov_sets(p_se,ys,qh)
        rows.append({"dataset":"UAV-CAS","held_out":str(F),"seed":seed,
            "shift_balacc":balanced_accuracy_score(yh,(p_sh>=.5).astype(int)),"seen_balacc":balanced_accuracy_score(ys,(p_se>=.5).astype(int)),
            "shift_ECE":top_label_ece(ch["temperature"],yh,CFG["nbins"]),"seen_ECE":top_label_ece(cs["temperature"],ys,CFG["nbins"]),
            "shift_Brier":brier_binary(ch["temperature"],yh),"shift_cov_marg":sh["marg"],"shift_cov_bal":sh["bal"],"seen_cov_marg":se["marg"],
            "shift_setsize":sh["size"],"shift_AURC":aurc(np.maximum(p_sh,1-p_sh),((p_sh>=.5).astype(int)==yh).astype(float))[0],
            "set_empty":sh["empty"],"set_normal_only":sh["normal_only"],"set_attack_only":sh["attack_only"],"set_both":sh["both"]})
        del S,clf; gc.collect()
    print(" UAV-CAS",F,"panel done")
P=pd.DataFrame(rows); P.to_csv(os.path.join(CFG["report_dir"],"14_uavcas_panel_raw.csv"),index=False)
ms=lambda g,col:"%.3f \u00b1 %.3f"%(g[col].mean(),g[col].std())
A=pd.DataFrame([{"held_out":F,"bal_acc":ms(g,"shift_balacc"),"ECE":ms(g,"shift_ECE"),"Brier":ms(g,"shift_Brier"),
    "cov_marg":ms(g,"shift_cov_marg"),"cov_bal":ms(g,"shift_cov_bal"),"set_size":ms(g,"shift_setsize"),"AURC":ms(g,"shift_AURC")}
    for F,g in P.groupby("held_out")])
print("\n=== UAV-CAS trust panel (mean +/- std over seeds; coverage target 0.90) ==="); print(A.to_string(index=False))
A.to_csv(os.path.join(CFG["report_dir"],"14_uavcas_panel.csv"),index=False)
print("\nin-distribution (seen) marginal coverage: %.3f +/- %.3f"%(P["seen_cov_marg"].mean(),P["seen_cov_marg"].std()))

 UAV-CAS DoS panel done
 UAV-CAS DDoS panel done
 UAV-CAS Blackhole panel done
 UAV-CAS Wormhole panel done
 UAV-CAS Replay panel done

=== UAV-CAS trust panel (mean +/- std over seeds; coverage target 0.90) ===
 held_out       bal_acc           ECE         Brier      cov_marg       cov_bal      set_size          AURC
Blackhole 0.504 ± 0.001 0.474 ± 0.003 0.451 ± 0.002 0.557 ± 0.010 0.672 ± 0.007 1.332 ± 0.014 0.672 ± 0.004
     DDoS 0.984 ± 0.001 0.128 ± 0.003 0.047 ± 0.001 0.999 ± 0.001 0.998 ± 0.001 1.265 ± 0.005 0.001 ± 0.000
      DoS 0.986 ± 0.002 0.230 ± 0.004 0.085 ± 0.002 0.998 ± 0.001 0.999 ± 0.000 1.463 ± 0.009 0.003 ± 0.001
   Replay 0.504 ± 0.002 0.047 ± 0.003 0.198 ± 0.001 0.879 ± 0.004 0.766 ± 0.007 1.522 ± 0.009 0.261 ± 0.004
 Wormhole 0.506 ± 0.001 0.442 ± 0.004 0.434 ± 0.003 0.546 ± 0.010 0.650 ± 0.007 1.295 ± 0.014 0.645 ± 0.002

in-distribution (seen) marginal coverage: 0.848 +/- 0.006


In [7]:
# Prediction-set decomposition on UAV-CAS
dec=P.groupby("held_out")[["set_empty","set_normal_only","set_attack_only","set_both","shift_cov_marg"]].mean().round(3)
dec.columns=["empty","{normal}","{attack}","{both}","coverage(marg)"]
print("UAV-CAS prediction-set composition on the shifted family:"); print(dec.to_string())
dec.to_csv(os.path.join(CFG["report_dir"],"14_uavcas_decomposition.csv"))

UAV-CAS prediction-set composition on the shifted family:
           empty  {normal}  {attack}  {both}  coverage(marg)
held_out                                                    
Blackhole    0.0     0.667     0.001   0.332           0.557
DDoS         0.0     0.182     0.554   0.265           0.999
DoS          0.0     0.363     0.174   0.463           0.998
Replay       0.0     0.477     0.001   0.522           0.879
Wormhole     0.0     0.705     0.001   0.295           0.546


In [ ]:
# Fragility mechanism on UAV-CAS: does attribution reversal predict the coverage collapse here too?
frag=[]
for F in fams:
    for seed in CFG["seeds"]:
        S=cas_prepare(df,F,seed); clf=fit(S["X_train"],S["y_train"],seed); ex=shap.TreeExplainer(clf)
        seen=[g for g in fams if g!=F]
        def smp(X,m,sd):
            idx=np.where(m)[0]
            if len(idx)>CFG["n_shap"]: idx=np.random.default_rng(sd).choice(idx,CFG["n_shap"],replace=False)
            return X[idx]
        m_ref=mean_shap(ex,smp(S["X_seen"],np.isin(S["fam_seen"],seen),seed)); m_held=mean_shap(ex,smp(S["X_shift"],S["fam_shift"]==F,seed))
        rv=reversal_vector(m_ref,m_held); ap=np.maximum(m_ref,0.0)
        frag.append({"held_out":str(F),"seed":seed,"frag_raw":float(rv.sum()),"frag_norm":float(rv.sum()/(ap.sum()+1e-12))})
        del S,clf,ex; gc.collect()
    print(" UAV-CAS",F,"fragility done")
FR=pd.DataFrame(frag); FR.to_csv(os.path.join(CFG["report_dir"],"14_uavcas_fragility_raw.csv"),index=False)
fa=FR.groupby("held_out").agg(frag_raw=("frag_raw","mean"),frag_raw_s=("frag_raw","std"),frag_norm=("frag_norm","mean")).round(3)
cov=P.groupby("held_out")["shift_cov_marg"].mean()
m=fa.join(cov); print("\nUAV-CAS fragility vs coverage:"); print(m.to_string())
if m["frag_raw"].std()>0: print("\nSpearman(fragility, marginal coverage) on UAV-CAS = %.3f"%spearmanr(m["frag_raw"],m["shift_cov_marg"]).correlation)
fa.to_csv(os.path.join(CFG["report_dir"],"14_uavcas_fragility.csv"))

 UAV-CAS DoS fragility done
 UAV-CAS DDoS fragility done
 UAV-CAS Blackhole fragility done
 UAV-CAS Wormhole fragility done


In [ ]:
!git add reports/ notebooks/
!git commit -m "14 UAV-CAS third dataset (classical, six single-family classes): trust panel + prediction-set decomposition + fragility mechanism, attack-family held out"
!git push origin main